# Board Generation and Piece Identity Detection

End-to-end demo of the sheet-identity pipeline:

- **Part 1 - Print Time:** generate board images with slot-grid labels and QR codes, save PNGs and config JSON to `data/aruco_boards/demo/`
- **Part 2 - Ingest Time:** reload config, run `SheetAruco.load_sheet()` on photos, inspect detected metadata and piece labels, visualise results


## Import


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from loguru import logger as lg
from rich import get_console
from rich import print as rprint
from rich.console import Console

# some magic to make rich work in jupyter
# https://github.com/Textualize/rich/issues/3483
# enable it for every cell output with %load_ext rich
console: Console = get_console()
console.is_jupyter = False

In [ ]:
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np

from snap_fit.aruco.board_image_composer import BoardImageComposer
from snap_fit.aruco.sheet_metadata import SheetMetadata
from snap_fit.aruco.sheet_metadata import SheetMetadataDecoder
from snap_fit.aruco.slot_grid import SlotGrid
from snap_fit.config.aruco.aruco_board_config import ArucoBoardConfig
from snap_fit.config.aruco.aruco_detector_config import ArucoDetectorConfig
from snap_fit.config.aruco.metadata_zone_config import MetadataZoneConfig
from snap_fit.config.aruco.metadata_zone_config import SlotGridConfig
from snap_fit.config.aruco.sheet_aruco_config import SheetArucoConfig
from snap_fit.params.snap_fit_params import get_snap_fit_params
from snap_fit.puzzle.sheet_aruco import SheetAruco
from snap_fit.puzzle.sheet_manager import SheetManager


## Params and config


In [ ]:
params = get_snap_fit_params()
rprint(params)
rprint("aruco_board_fol:", params.paths.aruco_board_fol)
rprint("data_fol:", params.paths.data_fol)


## Helper


In [ ]:
def show_img(img_bgr: np.ndarray, title: str = "", figsize: tuple = (5, 7)) -> None:
    """Display a BGR image inline via matplotlib."""
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(rgb)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


## Part 1: Generate Board Images (Print Time)

Generate several board images with embedded slot-grid labels and QR codes. Save the PNGs and config JSON
to `data/aruco_boards/{board_config_id}/` so the operator can print them and later reload the exact same
config during ingest.


### Step 1: Define Configuration

`ArucoBoardConfig` controls the ArUco marker ring.
`SlotGridConfig` sets how many piece-slot columns and rows appear inside the ring.
`MetadataZoneConfig` enables the QR strip and connects the slot grid.
`SheetArucoConfig` is the full ingest config (used in Part 2).


In [ ]:
TAG = "demo"
BOARD_CONFIG_ID = "demo"
TOTAL_SHEETS = 3

board_config = ArucoBoardConfig()  # default: 5x7 ring, 920x1320 px
slot_grid_config = SlotGridConfig(cols=4, rows=3)
metadata_zone = MetadataZoneConfig(
    enabled=True,
    qr_n_codes=3,
    qr_ecc="M",
    text_enabled=True,
    slot_grid=slot_grid_config,
)
sheet_aruco_config = SheetArucoConfig(
    detector=ArucoDetectorConfig(board=board_config),
    metadata_zone=metadata_zone,
)

rprint("Board dimensions (px):", board_config.board_dimensions())
rprint("board_config:", board_config)
rprint("metadata_zone:", metadata_zone)


### Step 2: Generate and Save Board Images

Each sheet gets a unique `SheetMetadata` (tag, index, total). `BoardImageComposer` renders
the ArUco ring, slot-grid labels, QR codes, and human-readable text into one BGR PNG.


In [ ]:
save_dir = params.paths.aruco_board_fol / BOARD_CONFIG_ID
save_dir.mkdir(parents=True, exist_ok=True)

composer = BoardImageComposer(board_config, metadata_zone)
generated_paths: list = []

for i in range(TOTAL_SHEETS):
    metadata = SheetMetadata(
        tag_name=TAG,
        sheet_index=i,
        total_sheets=TOTAL_SHEETS,
        board_config_id=BOARD_CONFIG_ID,
    )
    img = composer.compose(metadata)
    out_path = save_dir / f"sheet_{i:02d}.png"
    cv2.imwrite(str(out_path), img)
    generated_paths.append(out_path)
    lg.info(f"Saved {out_path.name}  ({img.shape[1]}x{img.shape[0]} px)")

lg.info(f"All {TOTAL_SHEETS} board images saved to: {save_dir}")


### Step 3: Save Config JSON

Save both the `ArucoBoardConfig` and the full `SheetArucoConfig` alongside the images
so ingest can reload the exact same config without hard-coding anything.


In [ ]:
# ArucoBoardConfig - standalone (board geometry only)
board_cfg_path = save_dir / f"{BOARD_CONFIG_ID}_ArucoBoardConfig.json"
board_cfg_path.write_text(json.dumps(board_config.model_dump(), indent=2))

# SheetArucoConfig - full ingest config (includes board + metadata_zone)
sheet_cfg_path = save_dir / f"{BOARD_CONFIG_ID}_SheetArucoConfig.json"
sheet_cfg_path.write_text(sheet_aruco_config.model_dump_json(indent=2))

rprint("Saved configs:")
rprint(f"  {board_cfg_path}")
rprint(f"  {sheet_cfg_path}")
rprint("\nSheetArucoConfig JSON preview:")
print(sheet_cfg_path.read_text())


### Step 4: Display Generated Board Images

Verify the ArUco ring, slot-grid labels, QR strip, and human-readable text all appear correctly before printing.


In [ ]:
for i, p in enumerate(generated_paths):
    img = cv2.imread(str(p))
    show_img(img, title=f"Sheet {i:02d}  |  {p.name}")


### Step 5: Verify QR Code Round-Trip

`SheetMetadataDecoder` reads QR codes from a raw (non-rectified) image. Running it
on a generated PNG confirms that the QR payload encodes and decodes correctly.


In [ ]:
decoder = SheetMetadataDecoder()

for i, p in enumerate(generated_paths):
    img = cv2.imread(str(p))
    meta = decoder.decode(img)
    if meta is not None:
        rprint(f"Sheet {i:02d}: [green]OK[/green]  {meta}")
    else:
        lg.warning(f"Sheet {i:02d}: QR not decoded")


### Step 6: Inspect Slot Grid

`SlotGrid` computes the centre pixel of every slot and generates human-readable labels
(`A1`, `B2`, ...) using the same `generate_label()` convention as `PuzzleGenerator`.


In [ ]:
slot_grid = SlotGrid(slot_grid_config, board_config)

rprint(f"Slot size: {slot_grid._slot_w:.0f} x {slot_grid._slot_h:.0f} px")
rprint(f"Label grid ({slot_grid_config.cols} cols x {slot_grid_config.rows} rows):")
for row in range(slot_grid_config.rows):
    labels = [
        slot_grid.label_for_slot(col, row) for col in range(slot_grid_config.cols)
    ]
    rprint("  " + "  ".join(labels))


In [ ]:
# Overlay slot centres on the first generated board image
img = cv2.imread(str(generated_paths[0]))
img_vis = img.copy()
for x, y in slot_grid.slot_centers():
    cv2.circle(img_vis, (x, y), 10, (0, 0, 255), -1)
show_img(img_vis, title="Slot centre positions (red dots)")


## Part 2: Ingest Photos (Photo to Data Time)

**Workflow:**

1. Print the generated board images on A4 paper.
2. Place physical puzzle pieces on the board and note which slot each occupies.
3. Photograph each sheet (any angle - the ArUco ring enables perspective correction).
4. Save photos to `data/demo/sheets/` (pattern: `*.jpg` or `*.png`).
5. Run the cells below to ingest and see detected piece labels.

For this demo the generated board PNGs are used as stand-ins when no real photos are available.


### Step 7: Reload Config from Disk

Load the `SheetArucoConfig` that was saved in Step 3. This is what an ingest script
would do: read the JSON from the board folder and reconstruct the loader without
hard-coding any parameters.


In [ ]:
cfg_loaded = SheetArucoConfig.model_validate_json(sheet_cfg_path.read_text())

### here we can override some config values if we want
cfg_loaded.min_area = 8_000

rprint("Loaded SheetArucoConfig:")
rprint(cfg_loaded)

sheet_aruco = SheetAruco(cfg_loaded)
lg.info("SheetAruco ready")


### Step 8: Demo Ingest (Stand-In)

Load one of the generated board PNGs as if it were a real photo. Since the image has
no physical pieces the piece list will be empty, but `sheet.metadata` will be populated
from the QR code and the slot grid will be ready for label assignment.


In [ ]:
demo_photo = generated_paths[0]
sheet = sheet_aruco.load_sheet(demo_photo)
sheet.build_pieces()

rprint(f"Sheet ID:          {sheet.sheet_id}")
rprint(f"Detected metadata: {sheet.metadata}")
rprint(f"Pieces detected:   {len(sheet.pieces)}")

if sheet.pieces:
    for piece in sheet.pieces[:10]:
        rprint(f"  piece {piece.piece_id.piece_id:>3}  label={piece.label!r}")
else:
    lg.info("No pieces detected (expected: stand-in board has no physical pieces)")


### Step 9: Real Photo Ingest

Place your photos in `data/demo/sheets/` (any image extension), then run the cell below.
`SheetManager.add_sheets()` globs the folder, calls `sheet_aruco.load_sheet()` on each file,
and stores the resulting `Sheet` objects keyed by their relative path.

Each detected piece will have `piece.label` set to the slot label (`A1`, `B3`, ...) derived
from its contour centroid position on the slot grid.


In [ ]:
photos_dir = params.paths.data_fol / TAG / "sheets"

if not photos_dir.exists():
    rprint(f"[yellow]Photos folder not found: {photos_dir}[/yellow]")
    rprint("Create it and place your photos there, then re-run this cell.")
else:
    manager = SheetManager()
    manager.add_sheets(
        photos_dir,
        pattern="*.jpg",
        loader_func=sheet_aruco.load_sheet,
    )
    rprint(f"Loaded {len(manager.sheets)} sheet(s)")
    for sid, s in manager.sheets.items():
        rprint(f"\n  Sheet {sid}")
        rprint(f"    metadata: {s.metadata}")
        rprint(f"    pieces:   {len(s.pieces)}")
        for p in s.pieces[:5]:
            rprint(f"      piece {p.piece_id.piece_id:>3}  label={p.label!r}")
        if len(s.pieces) > 5:
            rprint(f"      ... and {len(s.pieces) - 5} more")


### Step 10: Visualise Detections

Draw slot centres (grey rings) and detected piece centroids (green dots with label) on
the rectified sheet image. Works on any `Sheet` object - use the stand-in `sheet` here,
or swap in any sheet from the `manager` above.


In [ ]:
from snap_fit.puzzle.sheet import Sheet


def visualise_sheet(target_sheet: Sheet, grid: SlotGrid) -> np.ndarray:
    img = target_sheet.img_orig.copy()
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    # Slot centres - grey rings
    for x, y in grid.slot_centers():
        # UPDATE: slot centres are computed in the original board image, still
        # when the aruco border was present: recompute them to match the new cropped image
        cv2.circle(img, (x, y), 14, (160, 160, 160), 2)
        lg.debug(f"Slot centre at: ({x}, {y})")

    # Detected pieces - green dot + label
    for piece in target_sheet.pieces:
        cx, cy = piece.contour.centroid
        # UPDATE: centroids are computed in the cropped piece image
        label = piece.label or "?"
        cv2.circle(img, (cx, cy), 8, (0, 200, 0), -1)
        lg.debug(f"Piece {piece.piece_id.piece_id} at: ({cx}, {cy}), label={label!r}")
        cv2.putText(
            img,
            label,
            (cx + 12, cy + 6),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 200, 0),
            2,
        )
        break

    return img


sid, sheet = next(iter(manager.sheets.items()))
rprint(f"Sheet {sid}")

vis_grid = SlotGrid(cfg_loaded.metadata_zone.slot_grid, cfg_loaded.detector.board)
vis = visualise_sheet(sheet, vis_grid)
show_img(vis, title=f"Sheet: {sheet.sheet_id}  |  pieces detected: {len(sheet.pieces)}")
